In [1]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
import pandas as pd
import os
from tqdm.auto import tqdm
import numpy as np

from qick import *
from qick.pyro import make_proxy
from qick import QickConfig
from qick.asm_v2 import QickSpan, QickSweep1D
import Pyro4

Pyro4.config.SERIALIZER = "pickle"
Pyro4.config.PICKLE_PROTOCOL_VERSION = 4

ns_host = "192.168.10.179"
ns_port = 8888
proxy_name = "myqick"

soc, soccfg = make_proxy(ns_host=ns_host, ns_port=ns_port, proxy_name=proxy_name)
print(soccfg)

Pyro.NameServer PYRO:Pyro.NameServer@0.0.0.0:8888
myqick PYRO:obj_34518db0653f433dba1c9fdfb439d6ef@192.168.10.179:41689
QICK running on ZCU216, software version 0.2.381

Firmware configuration (built Tue Sep 10 16:13:40 2024):

	Global clocks (MHz): tProc dispatcher timing 430.080, RF reference 245.760
	Groups of related clocks: [tProc timing clock, DAC tile 0, DAC tile 1, DAC tile 3], [DAC tile 2], [ADC tile 2]

	16 signal generator channels:
	0:	axis_signal_gen_v6 - fs=9584.640 Msps, fabric=599.040 MHz
		envelope memory: 16384 complex samples (1.709 us)
		32-bit DDS, range=9584.640 MHz
		DAC tile 2, blk 0 is 0_230 on JHC3, or QICK box DAC port 8
	1:	axis_signal_gen_v6 - fs=9584.640 Msps, fabric=599.040 MHz
		envelope memory: 4096 complex samples (0.427 us)
		32-bit DDS, range=9584.640 MHz
		DAC tile 2, blk 1 is 1_230 on JHC4, or QICK box DAC port 9
	2:	axis_signal_gen_v6 - fs=9584.640 Msps, fabric=599.040 MHz
		envelope memory: 8192 complex samples (0.855 us)
		32-bit DDS, range=9584

In [2]:
from qick_workspace.newscrip.s008_T1_ge import T1
from qick_workspace.newscrip.s007_SpinEcho_ge import SpinEcho
from qick_workspace.newscrip.s006_Ramsey_ge import Ramsey
from qick_workspace.tools.system_tool import ExperimentConfig
from temploop import config_list

config_all = ExperimentConfig(config_list)


In [12]:
config_all.update('res.ro_length',5)
config_all.unified_config

{'name': ['Q1', 'Q2', 'Q3', 'Q4'],
 'res_ch': [0, 0, 0, 0],
 'qb_ch': [0, 0, 0, 0],
 'qb_ch_ef': [2, 2, 2, 2],
 'ro_ch': [0, 0, 0, 0],
 'res_freq_ge': [6720.4009, 6743, 6810.8901, 6883.7547],
 'res_length': 10,
 'res_gain_ge': [0.8, 0.9, 0.8, 0.8],
 'res_phase': [0, 0, 0, 0],
 'res_sigma': [0.005, 0.005, 0.005, 0.005],
 'ro_length': 5,
 'nqz_res': [2, 2, 2, 2],
 'chi': [False, False, False, False],
 'pulse_type': ['arb', 'arb', 'arb', 'arb'],
 'qb_freq_ge': [3620.63506, 4029.170438, 3127.600198, 3311.82327],
 'qb_mixer': [3620.625062, 4029.170438, 3127.600198, 3313.053269],
 'qb_gain_ge': [0.1, 0.1, 0.1, 0.1],
 'qb_phase': [0, 0, 0, 0],
 'sigma_ge': [0.01, 0.01, 0.05, 0.05],
 'pi_gain_ge': [0.691195, 0.451744, 0.159435, 0.347061],
 'pi2_gain_ge': [0.40945, 0.283355, 0.075459, 0.148747],
 'qb_flat_top_length_ge': [0.1, 0.1, 0.1, 0.1],
 'qb_freq_ef': [3409.397119, 4000, 4000, 3095.881349],
 'qb_mixer_ef': [3409.397119, 4000, 4000, 3095.881349],
 'qb_gain_ef': [0.1, 0.1, 0.1, 0.1],
 'qb_p

In [45]:
temperature = 210
avg=150

from time import sleep
sleep(20*60)

In [46]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import numpy as np


# --- 參數設定 ---

csv_dir = "csvfolder2"
filename = f"{csv_dir}/qubit_results_{temperature}mK.csv"
qubits = ["Q1", "Q2", "Q3", "Q4"]

# 自動建立資料夾
if not os.path.exists(csv_dir):
    os.makedirs(csv_dir)



# 關閉繪圖視窗
plt.ioff()
pbar = tqdm(qubits)

try:
    for q_name in pbar:
        pbar.set_description(f"Measuring {q_name}")
        
        # 每次重新取得 config 確保環境乾淨
        config_all = ExperimentConfig(config_list)
        run_cfg = config_all.get_qubit(q_name)

        # --- Ramsey ($T_{2}^*$) ---
        run_cfg.update([
            ("steps", 100), 
            ("wait_time", QickSweep1D("waitloop", 0.0, 1)), 
            ("ramsey_freq", 4)
        ])
        t2r = Ramsey(soc, soccfg, run_cfg)
        t2rfit, t2rerror = t2r.run(avg)
        t2r.saveLabber(qb_idx=q_name, config_all=config_all, title=f'{temperature}mK')
        plt.close('all')

        # --- Echo ($T_{2e}$) ---
        t2e = SpinEcho(soc, soccfg, run_cfg)
        t2efit, t2eerror = t2e.run(avg)
        t2e.saveLabber(qb_idx=q_name, config_all=config_all, title=f'{temperature}mK')
        plt.close('all')

        # --- T1 ---
        run_cfg.update([
            ("steps", 100), 
            ("wait_time", QickSweep1D("waitloop", 0.0, 10))
        ])
        t1 = T1(soc, soccfg, run_cfg)
        t1fit, t1error = t1.run(avg)
        t1.saveLabber(qb_idx=q_name, config_all=config_all, title=f'{temperature}mK')
        plt.close('all')

        # --- 數據整理 ---
        t2r_val, t2r_err = t2rfit[3], np.sqrt(np.diag(t2rerror))[3][3]
        t2e_val, t2e_err = t2efit[3], np.sqrt(np.diag(t2eerror))[3][3]
        t1_val, t1_err = t1fit[2], np.sqrt(np.diag(t1error))[2][2]

        data = {
            'qubit': q_name,
            't1_time': t1_val,
            't1_err': t1_err,
            't2r_time': t2r_val,
            't2r_err': t2r_err,
            't2e_time': t2e_val,
            't2e_err': t2e_err,
            'temp_mK': temperature,
            'timestamp': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
        }

        # --- 即時存檔 ---
        df_single = pd.DataFrame([data])
        file_exists = os.path.isfile(filename)
        df_single.to_csv(filename, mode='a', index=False, header=not file_exists)

        # 輸出進度，使用 .2f 格式化讓數值更易讀
        tqdm.write(f"✅ {q_name} | T1: {t1_val:.2f}±{t1_err:.2f} | T2e: {t2e_val:.2f}±{t2e_err:.2f} us")

except KeyboardInterrupt:
    tqdm.write("\n[!] 偵測到手動中斷。正在保留已寫入的數據...")
except Exception as e:
    tqdm.write(f"\n[X] 執行過程中發生錯誤: {e}")
finally:
    plt.ion()
    tqdm.write("\n量測任務結束。")

Data saved to D:\Labber_Data\Jay\purcell_tmon\temperature\2026\03\Data_0316\s008_T1_ge_Q4_210mK_001
✅ Q4 | T1: 1.15±0.41 | T2e: 0.94±0.86 us

量測任務結束。


In [48]:
config_all.unified_config

{'name': ['Q1', 'Q2', 'Q3', 'Q4'],
 'res_ch': [0, 0, 0, 0],
 'qb_ch': [0, 0, 0, 0],
 'qb_ch_ef': [2, 2, 2, 2],
 'ro_ch': [0, 0, 0, 0],
 'res_freq_ge': [6720.4009, 6743, 6810.8901, 6883.7547],
 'res_length': 10,
 'res_gain_ge': [0.8, 0.9, 0.8, 0.8],
 'res_phase': [0, 0, 0, 0],
 'res_sigma': [0.005, 0.005, 0.005, 0.005],
 'ro_length': 5,
 'nqz_res': [2, 2, 2, 2],
 'chi': [False, False, False, False],
 'pulse_type': ['arb', 'arb', 'arb', 'arb'],
 'qb_freq_ge': [3620.63506, 4029.170438, 3127.600198, 3311.82327],
 'qb_mixer': [3620.625062, 4029.170438, 3127.600198, 3313.053269],
 'qb_gain_ge': [0.1, 0.1, 0.1, 0.1],
 'qb_phase': [0, 0, 0, 0],
 'sigma_ge': [0.01, 0.01, 0.05, 0.05],
 'pi_gain_ge': [0.691195, 0.451744, 0.159435, 0.347061],
 'pi2_gain_ge': [0.40945, 0.283355, 0.075459, 0.148747],
 'qb_flat_top_length_ge': [0.1, 0.1, 0.1, 0.1],
 'qb_freq_ef': [3409.397119, 4000, 4000, 3095.881349],
 'qb_mixer_ef': [3409.397119, 4000, 4000, 3095.881349],
 'qb_gain_ef': [0.1, 0.1, 0.1, 0.1],
 'qb_p